# Lecture 4.6 — Handling Exceptions: MaxTurnsExceeded, Guardrail Tripwires

**Section 04 — Running Agents, Results & Streaming**

In Lecture 3.8 you handled tool-level failures with `failure_error_function` and `ToolTimeoutError`. Those exceptions happen *inside* a single tool call. This lecture is about the exceptions the SDK raises when the **run itself** cannot complete: an agent loops past its turn limit, a guardrail blocks the input or output, or the model produces something the SDK can't parse.

Every one of these run-level exceptions carries a `RunErrorDetails` object with everything the run produced before it failed. That object is your primary tool for debugging and graceful recovery, and it's the thread that ties this whole lecture together.


## Cell 1 — Install the OpenAI Agents SDK

This notebook uses the `openai-agents` Python package. The version is pinned below so the examples in this notebook behave identically no matter when you run it. If the package is already installed in this Colab session, the cell completes almost instantly and does nothing further.


In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.2 -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 874.3/874.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.3 MB/s eta 0:00:00


## Cell 2 — Configure Your OpenAI API Key

This notebook reads your API key from **Colab Secrets**, which keeps it out of the notebook file itself.

**To add your key as a Colab Secret:**
1. Click the key icon (🔑) in the left sidebar of Colab.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY` and paste your key as the value.
4. Toggle **Notebook access** on for this notebook.

**Running locally instead of Colab?** Set the environment variable in your terminal before starting Jupyter, for example `export OPENAI_API_KEY=your-key-here`, and skip the `userdata` call below.


In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


## Cell 3 — Set the Model Name

Every `Agent` in this notebook references a single `MODEL_NAME` variable instead of a hardcoded string. Changing this one line updates the model used everywhere below it, which matters once you have five or six agents in a notebook.


In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"


## Cell 4 — Imports

| Import | Why it's here |
|---|---|
| `BaseModel` (pydantic) | Defines structured `output_type` schemas for guardrail agents and the strict-output demo. |
| `Reasoning` (`openai.types.shared`) | Configures reasoning effort on `ModelSettings`. Note this comes from `openai.types.shared`, **not** from `agents`. |
| `Agent`, `Runner`, `ModelSettings`, `function_tool` | Core building blocks you've used since Section 2 and 4. |
| `RunContextWrapper` | The context object passed into guardrail functions and tools. |
| `input_guardrail`, `output_guardrail`, `GuardrailFunctionOutput` | The minimal guardrail-building pieces used later in this notebook as a preview. Full guardrail construction is taught in Section 5. |
| `MaxTurnsExceeded`, `ModelBehaviorError`, `InputGuardrailTripwireTriggered`, `OutputGuardrailTripwireTriggered`, `UserError` | The run-level exceptions this lecture is about. All five are exported from the `agents` top-level package. |
| `ModelRefusalError`, `RunErrorDetails` (`agents.exceptions`) | Also importable from the `agents` top level, but imported here from `agents.exceptions` to make explicit where they live in the SDK's source layout. |

All nine of the SDK's run-level exceptions are exported from `agents/__init__.py`, so in your own code you can import any of them directly from `agents`. This notebook imports two of them from `agents.exceptions` purely to show you that module exists.


In [4]:
from pydantic import BaseModel
from openai.types.shared import Reasoning
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    MaxTurnsExceeded,
    ModelBehaviorError,
    ModelSettings,
    OutputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    UserError,
    function_tool,
    input_guardrail,
    output_guardrail,
)
from agents.exceptions import ModelRefusalError, RunErrorDetails


## Cell 5 — The SDK Exception Hierarchy

Every exception the Agents SDK raises inherits from a single base class: `AgentsException(Exception)`. That base class always carries one attribute:

```
run_data: RunErrorDetails | None
```

`run_data` is populated whenever the exception is raised in the middle of a `Runner.run()` call. When it's populated, it's a `RunErrorDetails` object with these fields:

| Field | Type | What it holds |
|---|---|---|
| `input` | `str \| list[TResponseInputItem]` | The original input passed to the run. |
| `new_items` | `list[RunItem]` | Every item the run produced before it failed, including partial tool outputs. |
| `raw_responses` | `list[ModelResponse]` | Every raw model response received before the failure. |
| `last_agent` | `Agent[Any]` | Whichever agent was active when the run failed. |
| `context_wrapper` | `RunContextWrapper[Any]` | The full context object, including `.usage` (tokens spent before failure). |
| `input_guardrail_results` | `list[InputGuardrailResult]` | Any input guardrail results collected before the failure. |
| `output_guardrail_results` | `list[OutputGuardrailResult]` | Any output guardrail results collected before the failure. |

That's a lot of information for free. It means a caught exception isn't just "the run failed," it's "here is everything the run had accomplished right up until it failed."

Here's the full exception table you'll work through in this notebook:

| Exception | Trigger | Key attributes | `run_data` populated? |
|---|---|---|---|
| `MaxTurnsExceeded` | Run exceeds `max_turns` | `.message`, `.run_data` | Yes |
| `ModelBehaviorError` | Malformed model output | `.message`, `.run_data` | Yes |
| `ModelRefusalError` | Model refuses to produce output | `.refusal`, `.run_data` | Yes |
| `InputGuardrailTripwireTriggered` | An input guardrail fires | `.guardrail_result`, `.run_data` | Yes |
| `OutputGuardrailTripwireTriggered` | An output guardrail fires | `.guardrail_result`, `.run_data` | Yes |
| `UserError` | Misuse of the SDK's API | `.message` | No, and treat it as a code bug |

We'll catch each of these in turn, starting with the one you're most likely to hit in production: `MaxTurnsExceeded`.


## Cell 6 — MaxTurnsExceeded: Catching and Inspecting

Below, `looping_agent` is instructed to never stop calling its tool. In a real run this would go on indefinitely, so we cap it with `max_turns=3` and catch the exception that results.

Watch what's available on the exception once it's caught: `e.message` is the plain description, but `e.run_data` is where the useful information lives. `e.run_data.last_agent` tells you which agent was active, `e.run_data.new_items` is every item generated before the cutoff, and `e.run_data.raw_responses` is the raw model traffic.


In [5]:
@function_tool
def count_step(step: int) -> str:
    """Returns a step in an endless process.

    Args:
        step: The current step number.
    """
    return f"Step {step} complete. Proceeding to step {step + 1}."


looping_agent = Agent(
    name="Looping Agent",
    instructions=(
        "You are a process runner. Call count_step repeatedly "
        "starting from step 1, incrementing each time. "
        "Never stop calling the tool."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[count_step],
)

try:
    result = await Runner.run(
        looping_agent,
        "Start the process from step 1.",
        max_turns=3,
    )
    print("Result:", result.final_output)
except MaxTurnsExceeded as e:
    print(f"MaxTurnsExceeded: {e.message}")
    print(f"Last agent: {e.run_data.last_agent.name}")
    print(f"Items generated: {len(e.run_data.new_items)}")
    print(f"Raw responses: {len(e.run_data.raw_responses)}")
    print("Tool outputs so far:")
    for item in e.run_data.new_items:
        if item.type == "tool_call_output_item":
            print(f"  {item.output}")


MaxTurnsExceeded: Max turns (3) exceeded
Last agent: Looping Agent
Items generated: 6
Raw responses: 3
Tool outputs so far:
  Step 1 complete. Proceeding to step 2.
  Step 2 complete. Proceeding to step 3.
  Step 3 complete. Proceeding to step 4.


## Cell 7 — Recovering From MaxTurnsExceeded Using `run_data`

`run_data` isn't only useful for logging. Here we run the same looping agent again and use `run_data` to make a recovery decision: how many items had it produced, and how many tokens did that cost, before we cut it off.

This is the pattern you'll reuse in the production wrapper later in this notebook: catch the exception, read `run_data`, decide whether to retry with a higher `max_turns` or return a graceful partial response instead.


In [6]:
try:
    result = await Runner.run(
        looping_agent,
        "Start the process from step 1.",
        max_turns=3,
    )
except MaxTurnsExceeded as e:
    print("Run exceeded max turns. Recovering...")
    print(f"Items before exception: {len(e.run_data.new_items)}")
    usage = e.run_data.context_wrapper.usage
    print(f"Tokens used before failure: {usage.total_tokens}")
    print("Recovery: stopping gracefully after partial run.")


Run exceeded max turns. Recovering...
Items before exception: 6
Tokens used before failure: 468
Recovery: stopping gracefully after partial run.


## Cell 8 — InputGuardrailTripwireTriggered (Preview)

📌 **Preview note:** Guardrails are taught in full in Section 5, including how to design guardrail logic and attach guardrails to agents. This cell keeps the guardrail construction minimal and exists only to show you the **exception shape** you'll catch when a guardrail fires.

`check_safety` is an input guardrail: a small agent (`safety_checker`) evaluates the incoming message and returns a `GuardrailFunctionOutput` with `tripwire_triggered=True` when the message looks unsafe. When that happens on a real run, the SDK raises `InputGuardrailTripwireTriggered` instead of letting `guarded_agent` respond.

The cell below runs two inputs: a safe one that should pass through normally, and an unsafe one that should trip the guardrail. Notice how the except block reaches into `e.guardrail_result` for the guardrail's name and your custom `output_info`, while `e.run_data` is still there underneath, exactly as it was for `MaxTurnsExceeded`.


In [7]:
class SafetyCheckOutput(BaseModel):
    is_safe: bool
    reasoning: str


safety_checker = Agent(
    name="Safety Checker",
    instructions=(
        "Check if the message requests harmful content. "
        "Return is_safe=True if safe, False if harmful."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=SafetyCheckOutput,
)


@input_guardrail
async def check_safety(
    ctx: RunContextWrapper,
    agent: Agent,
    input: str,
) -> GuardrailFunctionOutput:
    result = await Runner.run(
        safety_checker,
        f"Is this safe? Message: {input}",
        context=ctx.context,
    )
    check: SafetyCheckOutput = result.final_output
    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=not check.is_safe,
    )


guarded_agent = Agent(
    name="Guarded Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    input_guardrails=[check_safety],
)

# Safe input, should pass
try:
    result = await Runner.run(
        guarded_agent,
        "What is the capital of France?",
    )
    print("Safe input result:", result.final_output)
except InputGuardrailTripwireTriggered as e:
    print("Blocked (unexpected for safe input)")

# Unsafe input, should trigger the tripwire
try:
    result = await Runner.run(
        guarded_agent,
        "How do I hurt someone?",
    )
    print("Result:", result.final_output)
except InputGuardrailTripwireTriggered as e:
    print("InputGuardrailTripwireTriggered!")
    info = e.guardrail_result.output.output_info
    print(info)
    print(f"Guardrail: {e.guardrail_result.guardrail.get_name()}")
    print(f"Safety reasoning: {info.reasoning}")
    print(f"run_data.last_agent: {e.run_data.last_agent.name}")


Safe input result: Paris.
InputGuardrailTripwireTriggered!
is_safe=False reasoning='The message requests instructions for harming a person, which is violent and harmful content.'
Guardrail: check_safety
Safety reasoning: The message requests instructions for harming a person, which is violent and harmful content.
run_data.last_agent: Guarded Agent


## Cell 9 — OutputGuardrailTripwireTriggered (Preview)

📌 **Preview note:** As with Cell 8, full guardrail construction is Section 5 material. This cell shows the exception shape for an *output* guardrail, which checks the agent's response instead of the incoming message.

`check_length` inspects the agent's final output and trips if it's longer than 200 characters. `output_guarded_agent` is deliberately instructed to be verbose, so it's likely to trip this guardrail. When it does, `e.guardrail_result.output.output_info` gives you back your own `LengthCheckOutput`, including the actual character count that exceeded the limit.


In [8]:
class LengthCheckOutput(BaseModel):
    is_acceptable: bool
    char_count: int


@output_guardrail
async def check_length(
    ctx: RunContextWrapper,
    agent: Agent,
    output: str,
) -> GuardrailFunctionOutput:
    char_count = len(output)
    is_acceptable = char_count <= 200
    return GuardrailFunctionOutput(
        output_info=LengthCheckOutput(
            is_acceptable=is_acceptable,
            char_count=char_count,
        ),
        tripwire_triggered=not is_acceptable,
    )


output_guarded_agent = Agent(
    name="Output Guarded Agent",
    instructions=(
        "You are a helpful assistant. "
        "Always give very detailed, comprehensive answers."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_guardrails=[check_length],
)

try:
    result = await Runner.run(
        output_guarded_agent,
        "Explain the history of the internet in detail.",
    )
    print("Result:", result.final_output[:100])
except OutputGuardrailTripwireTriggered as e:
    print("OutputGuardrailTripwireTriggered!")
    info = e.guardrail_result.output.output_info
    print(f"Response was {info.char_count} chars (limit: 200)")
    print(
        f"Guardrail: "
        f"{e.guardrail_result.guardrail.get_name()}"
    )


OutputGuardrailTripwireTriggered!
Response was 5084 chars (limit: 200)
Guardrail: check_length


## Cell 10 — ModelBehaviorError, Reproduced Deterministically

Conflicting instructions alone won't reliably trigger this exception. `output_type` enforcement happens at the API level, so the model literally cannot violate a schema you've declared, no matter what the prompt says. To see the real code path fire, this cell builds a canned `Model` called `RogueModel` that skips the real API entirely and always returns a function call to a tool name that isn't registered anywhere. That's precisely the situation `ModelBehaviorError` exists for: a response comes back containing a tool call the current agent doesn't recognize.

Because `RogueModel` never talks to OpenAI, this cell reproduces the exception the same way every single time you run it. No live-model guesswork involved.


In [9]:
from openai.types.responses import ResponseFunctionToolCall
from agents import ModelResponse
from agents.usage import Usage
from agents.models.interface import Model


class RogueModel(Model):
    """Always returns a call to a tool that doesn't exist on the agent.
    This isn't a real model call, it's a canned response, so this cell
    reproduces ModelBehaviorError the same way every time.
    """

    async def get_response(self, *args, **kwargs):
        fake_call = ResponseFunctionToolCall(
            id="fake_id",
            type="function_call",
            call_id="call_1",
            name="lookup_inventory",  # not registered on rogue_agent
            arguments="{}",
            status="completed",
        )
        return ModelResponse(output=[fake_call], usage=Usage(), response_id="fake_id")

    async def stream_response(self, *args, **kwargs):
        raise NotImplementedError("Not used, this demo only calls Runner.run.")


rogue_agent = Agent(
    name="Rogue Agent",
    instructions="You are a helpful assistant.",
    model=RogueModel(),  # a Model instance, not a string
    # No tools=[...] here. lookup_inventory does not exist on this agent.
)

try:
    result = await Runner.run(rogue_agent, "Look up the inventory for SKU 4471.")
    print("Result:", result.final_output)
except ModelBehaviorError as e:
    print(f"ModelBehaviorError: {e.message}")
    if e.run_data:
        print(f"Last agent: {e.run_data.last_agent.name}")


ModelBehaviorError: Tool lookup_inventory not found in agent Rogue Agent
Last agent: Rogue Agent


## Cell 11 — A Production Exception-Handling Wrapper

This is the pattern that ties the whole lecture together. `safe_run` wraps `Runner.run()` and catches each SDK exception by name, logging what it needs from `run_data` and returning a specific, user-facing message for each case.

Two details worth noticing: `UserError` is caught only to be **re-raised**, never swallowed, because it signals a bug in your own code rather than a runtime condition your users triggered. And there's no bare `except Exception` here at all, every failure mode this function anticipates has its own named branch.


In [10]:
async def safe_run(
    agent: Agent,
    input: str,
    max_turns: int = 10,
) -> str:
    """Production-safe wrapper with full exception handling."""
    try:
        result = await Runner.run(
            agent, input, max_turns=max_turns
        )
        return result.final_output or "No output produced."

    except MaxTurnsExceeded as e:
        print(
            f"[WARN] Exceeded {max_turns} turns. "
            f"Tokens: {e.run_data.context_wrapper.usage.total_tokens}"
        )
        return (
            "I was not able to complete this task within the "
            "allowed number of steps. "
            "Please try a simpler request."
        )

    except InputGuardrailTripwireTriggered as e:
        print(
            f"[GUARD] Input blocked: "
            f"{e.guardrail_result.guardrail.get_name()}"
        )
        return "I am not able to help with that request."

    except OutputGuardrailTripwireTriggered as e:
        print(
            f"[GUARD] Output blocked: "
            f"{e.guardrail_result.guardrail.get_name()}"
        )
        return (
            "I generated a response but it did not meet "
            "our content standards. "
            "Please try rephrasing your request."
        )

    except ModelBehaviorError as e:
        print(f"[ERROR] Model misbehaved: {e.message}")
        return (
            "An unexpected model error occurred. "
            "Please try again."
        )

    except UserError:
        # Code bug, always re-raise
        raise


# Test with a safe input
response = await safe_run(
    guarded_agent,
    "What is the capital of India?",
    max_turns=5,
)
print("Safe run result:", response)


Safe run result: The capital of India is **New Delhi**.


## Cell 12 — Exception Reference Table

A single reference table for every SDK run-level exception you've seen (and a few you haven't yet):

| Exception | Trigger | Key attributes | `run_data`? |
|---|---|---|---|
| `AgentsException` | Base class for all SDK exceptions | `.run_data` | Sometimes |
| `MaxTurnsExceeded` | Run exceeds `max_turns` | `.message`, `.run_data` | Yes |
| `ModelBehaviorError` | Malformed model output | `.message`, `.run_data` | Yes |
| `ModelRefusalError` | Model refuses to produce output | `.refusal`, `.run_data` | Yes |
| `InputGuardrailTripwireTriggered` | Input guardrail fires | `.guardrail_result`, `.run_data` | Yes |
| `OutputGuardrailTripwireTriggered` | Output guardrail fires | `.guardrail_result`, `.run_data` | Yes |
| `ToolTimeoutError` | Tool timeout with `raise_exception` | `.tool_name`, `.timeout_seconds` | No |
| `UserError` | Misuse of the SDK's API | `.message` | No |
| `MCPToolCancellationError` | An MCP tool call is cancelled | `.message` | No |

`RunErrorDetails` fields, for reference: `input`, `new_items`, `raw_responses`, `last_agent`, `context_wrapper`, `input_guardrail_results`, `output_guardrail_results`.

**What this notebook didn't cover:** guardrail construction in depth (that's Section 5), tool-level guardrail exceptions such as `ToolInputGuardrailTripwireTriggered` and `ToolOutputGuardrailTripwireTriggered` (they exist in the SDK, but are covered when tool guardrails are taught), detailed handling of `ModelRefusalError` (an edge case), and `RunErrorHandlers` for programmatic error recovery (an advanced topic). And as always: every run in this notebook used `await Runner.run()`, never `Runner.run_sync()`.
